# 03 — Data-Quality Validation

Standardizes values, evaluates deterministic rules, quarantines invalid and duplicate records, and produces Silver data.

## Publication notes

- This public notebook contains no credentials, tokens, tenant IDs, subscription IDs, or workspace URLs.
- Notebook outputs were removed to avoid publishing source records or environment metadata.
- Configure the catalog, schema, and source data location for your own Azure environment before execution.
- Run notebooks in numerical order.


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import uuid

CATALOG = "dbx_niw_analytics"
SCHEMA = "ztlf"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_customer_churn_experiment"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_customer_churn"
QUARANTINE_TABLE = f"{CATALOG}.{SCHEMA}.customer_churn_quarantine"
METRICS_TABLE = f"{CATALOG}.{SCHEMA}.experiment_metrics"

QUALITY_RUN_ID = str(uuid.uuid4())

print(f"Quality run ID: {QUALITY_RUN_ID}")
print(f"Source table: {SOURCE_TABLE}")

In [ ]:
dirty_df = spark.table(SOURCE_TABLE)

dirty_count = dirty_df.count()

print(f"Dirty dataset rows: {dirty_count}")
display(dirty_df.limit(10))

In [ ]:
VALID_GEOGRAPHIES = ["France", "Germany", "Spain"]
VALID_GENDERS = ["Male", "Female"]

MANDATORY_COLUMNS = [
    "CustomerId",
    "CreditScore",
    "Geography",
    "Gender",
    "Age",
    "Balance",
    "EstimatedSalary"
]

In [ ]:
standardized_df = (
    dirty_df
    .withColumn(
        "Geography",
        F.when(
            F.lower(F.trim(F.col("Geography"))) == "france",
            F.lit("France")
        )
        .when(
            F.lower(F.trim(F.col("Geography"))) == "germany",
            F.lit("Germany")
        )
        .when(
            F.lower(F.trim(F.col("Geography"))) == "spain",
            F.lit("Spain")
        )
        .otherwise(F.trim(F.col("Geography")))
    )
    .withColumn(
        "Gender",
        F.when(
            F.lower(F.trim(F.col("Gender"))).isin("male", "m"),
            F.lit("Male")
        )
        .when(
            F.lower(F.trim(F.col("Gender"))).isin("female", "f"),
            F.lit("Female")
        )
        .otherwise(F.trim(F.col("Gender")))
    )
)

In [ ]:
validated_df = (
    standardized_df
    .withColumn(
        "_rule_customer_id_valid",
        (
            F.col("CustomerId").isNotNull()
            & (F.col("CustomerId") > 0)
        )
    )
    .withColumn(
        "_rule_credit_score_valid",
        (
            F.col("CreditScore").isNotNull()
            & F.col("CreditScore").between(300, 850)
        )
    )
    .withColumn(
        "_rule_age_valid",
        (
            F.col("Age").isNotNull()
            & F.col("Age").between(18, 100)
        )
    )
    .withColumn(
        "_rule_balance_valid",
        (
            F.col("Balance").isNotNull()
            & (F.col("Balance") >= 0)
        )
    )
    .withColumn(
        "_rule_salary_valid",
        (
            F.col("EstimatedSalary").isNotNull()
            & (F.col("EstimatedSalary") >= 0)
        )
    )
    .withColumn(
        "_rule_geography_valid",
        F.col("Geography").isin(VALID_GEOGRAPHIES)
    )
    .withColumn(
        "_rule_gender_valid",
        F.col("Gender").isin(VALID_GENDERS)
    )
)

In [ ]:
validated_df = validated_df.withColumn(
    "_quality_errors",
    F.filter(
        F.array(
            F.when(
                ~F.col("_rule_customer_id_valid"),
                F.lit("INVALID_CUSTOMER_ID")
            ),
            F.when(
                ~F.col("_rule_credit_score_valid"),
                F.lit("INVALID_CREDIT_SCORE")
            ),
            F.when(
                ~F.col("_rule_age_valid"),
                F.lit("INVALID_AGE")
            ),
            F.when(
                ~F.col("_rule_balance_valid"),
                F.lit("INVALID_BALANCE")
            ),
            F.when(
                ~F.col("_rule_salary_valid"),
                F.lit("INVALID_OR_MISSING_SALARY")
            ),
            F.when(
                ~F.col("_rule_geography_valid"),
                F.lit("INVALID_GEOGRAPHY")
            ),
            F.when(
                ~F.col("_rule_gender_valid"),
                F.lit("INVALID_GENDER")
            )
        ),
        lambda error: error.isNotNull()
    )
)

validated_df = validated_df.withColumn(
    "_record_quality_status",
    F.when(
        F.size(F.col("_quality_errors")) == 0,
        F.lit("VALID")
    ).otherwise(F.lit("INVALID"))
)

In [ ]:
display(
    validated_df.groupBy("_record_quality_status").count()
)

In [ ]:
display(
    validated_df
    .filter(F.col("_record_quality_status") == "INVALID")
    .select(
        "CustomerId",
        "CreditScore",
        "Geography",
        "Gender",
        "Age",
        "Balance",
        "EstimatedSalary",
        "_quality_errors"
    )
    .limit(50)
)

In [ ]:
quarantine_df = (
    validated_df
    .filter(F.col("_record_quality_status") == "INVALID")
    .withColumn("_quarantine_timestamp", F.current_timestamp())
    .withColumn("_quality_run_id", F.lit(QUALITY_RUN_ID))
)

valid_df = validated_df.filter(
    F.col("_record_quality_status") == "VALID"
)

print(f"Valid rows before deduplication: {valid_df.count()}")
print(f"Quarantined rows: {quarantine_df.count()}")

In [ ]:
dedupe_window = Window.partitionBy("CustomerId").orderBy(
    F.col("_is_injected_duplicate").asc(),
    F.col("_experiment_row_number").asc()
)

deduplicated_df = (
    valid_df
    .withColumn(
        "_duplicate_rank",
        F.row_number().over(dedupe_window)
    )
)

duplicate_reject_df = (
    deduplicated_df
    .filter(F.col("_duplicate_rank") > 1)
    .withColumn(
        "_quality_errors",
        F.array(F.lit("DUPLICATE_CUSTOMER_ID"))
    )
    .withColumn(
        "_record_quality_status",
        F.lit("INVALID")
    )
    .withColumn(
        "_quarantine_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "_quality_run_id",
        F.lit(QUALITY_RUN_ID)
    )
)

silver_candidate_df = deduplicated_df.filter(
    F.col("_duplicate_rank") == 1
)

all_quarantine_df = quarantine_df.unionByName(
    duplicate_reject_df,
    allowMissingColumns=True
)

In [ ]:
columns_to_drop = [
    "_duplicate_rank",
    "_is_injected_duplicate",
    "_experiment_row_number",
    "_dataset_type",
    "_created_timestamp",
    "_processing_layer",
    "_record_status",
    "_rule_customer_id_valid",
    "_rule_credit_score_valid",
    "_rule_age_valid",
    "_rule_balance_valid",
    "_rule_salary_valid",
    "_rule_geography_valid",
    "_rule_gender_valid",
    "_quality_errors",
    "_record_quality_status"
]

silver_df = (
    silver_candidate_df
    .drop(*columns_to_drop)
    .withColumn("_silver_processed_at", F.current_timestamp())
    .withColumn("_quality_run_id", F.lit(QUALITY_RUN_ID))
    .withColumn("_processing_layer", F.lit("SILVER"))
    .withColumn("_record_status", F.lit("VALIDATED"))
)

print(f"Silver rows: {silver_df.count()}")
display(silver_df.limit(10))

In [ ]:
before_metrics = dirty_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(
        F.col("_is_injected_duplicate").cast("int")
    ).alias("duplicate_rows"),
    F.sum(
        F.col("EstimatedSalary").isNull().cast("int")
    ).alias("missing_salary_rows"),
    F.sum(
        (
            (F.col("Age") < 18)
            | (F.col("Age") > 100)
        ).cast("int")
    ).alias("invalid_age_rows"),
    F.sum(
        (F.col("Balance") < 0).cast("int")
    ).alias("negative_balance_rows"),
    F.sum(
        (
            F.col("CustomerId").isNull()
            | (F.col("CustomerId") <= 0)
        ).cast("int")
    ).alias("invalid_customer_id_rows")
).first().asDict()

after_metrics = silver_df.agg(
    F.count("*").alias("total_rows"),
    (
        F.count("*")
        - F.countDistinct("CustomerId")
    ).alias("duplicate_rows"),
    F.sum(
        F.col("EstimatedSalary").isNull().cast("int")
    ).alias("missing_salary_rows"),
    F.sum(
        (
            (F.col("Age") < 18)
            | (F.col("Age") > 100)
        ).cast("int")
    ).alias("invalid_age_rows"),
    F.sum(
        (F.col("Balance") < 0).cast("int")
    ).alias("negative_balance_rows"),
    F.sum(
        (
            F.col("CustomerId").isNull()
            | (F.col("CustomerId") <= 0)
        ).cast("int")
    ).alias("invalid_customer_id_rows")
).first().asDict()

comparison_rows = []

for metric_name in before_metrics:
    comparison_rows.append(
        (
            metric_name,
            int(before_metrics[metric_name] or 0),
            int(after_metrics[metric_name] or 0)
        )
    )

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["metric_name", "before_value", "after_value"]
)

comparison_df = comparison_df.withColumn(
    "improvement",
    F.col("before_value") - F.col("after_value")
)

display(comparison_df)

In [ ]:
bronze_scored_df = validated_df.withColumn(
    "_valid_rule_count",
    F.col("_rule_customer_id_valid").cast("int")
    + F.col("_rule_credit_score_valid").cast("int")
    + F.col("_rule_age_valid").cast("int")
    + F.col("_rule_balance_valid").cast("int")
    + F.col("_rule_salary_valid").cast("int")
    + F.col("_rule_geography_valid").cast("int")
    + F.col("_rule_gender_valid").cast("int")
)

bronze_quality_score = (
    bronze_scored_df
    .select(
        (
            F.avg("_valid_rule_count") / F.lit(7.0) * 100
        ).alias("quality_score")
    )
    .first()["quality_score"]
)

silver_quality_score = (
    silver_candidate_df
    .select(
        (
            F.avg(
                F.col("_rule_customer_id_valid").cast("int")
                + F.col("_rule_credit_score_valid").cast("int")
                + F.col("_rule_age_valid").cast("int")
                + F.col("_rule_balance_valid").cast("int")
                + F.col("_rule_salary_valid").cast("int")
                + F.col("_rule_geography_valid").cast("int")
                + F.col("_rule_gender_valid").cast("int")
            ) / F.lit(7.0) * 100
        ).alias("quality_score")
    )
    .first()["quality_score"]
)

quality_score_df = spark.createDataFrame(
    [
        ("BRONZE_EXPERIMENT", float(bronze_quality_score)),
        ("SILVER", float(silver_quality_score))
    ],
    ["dataset_stage", "quality_score_percent"]
)

display(quality_score_df)

In [ ]:
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Silver table created: {SILVER_TABLE}")

In [ ]:
(
    all_quarantine_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE_TABLE)
)

print(f"Quarantine table created: {QUARANTINE_TABLE}")

In [ ]:
display(spark.table(SILVER_TABLE).limit(20))

In [ ]:
display(spark.table(QUARANTINE_TABLE).limit(20))